<a href="https://colab.research.google.com/github/OjaswiGautam/FlyrankAI/blob/main/work/notebooks/Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import os
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

TABLE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

In [21]:
raw = con.sql(f"""
    WITH scoped AS (
        SELECT client_hash_id, content_hash_id, gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet('{TABLE}')
        WHERE gsc_data_available = TRUE
    ),
    agg AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(gsc_impressions) AS gsc_impressions,
            SUM(gsc_clicks) AS gsc_clicks,
            SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position
        FROM scoped
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT a.*, d.content_created_date, d.word_count
    FROM agg a
    LEFT JOIN read_parquet('{DIM}') d
      ON a.client_hash_id = d.client_hash_id
     AND a.content_hash_id = d.content_hash_id
    ORDER BY a.client_hash_id, a.content_hash_id
""").df()

df = raw.copy()
df['content_age_days'] = (pd.Timestamp('2026-03-31') - pd.to_datetime(df['content_created_date'])).dt.days

print("Modeling population shape:", df.shape)
print("Unique content_hash_id:", df['content_hash_id'].nunique())
print("Unique client_hash_id:", df['client_hash_id'].nunique())
print("Duplicate (client, content) pairs:",
      df.duplicated(['client_hash_id', 'content_hash_id']).sum())
print("\nMissingness per modeling feature:")
print(df[['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'content_age_days', 'word_count']].isna().mean().round(4))
print("\navg_position == 0 count:", (df['gsc_avg_position'] == 0).sum(),
      f"({(df['gsc_avg_position'] == 0).mean()*100:.2f}%)")

# Reproducibility check — first and last 3 rows, so we can visually confirm
# the same order comes back if this cell is re-run
print("\nFirst 3 rows (client_hash_id, content_hash_id) — for reproducibility verification:")
print(df[['client_hash_id', 'content_hash_id']].head(3).to_string(index=False))
print("\nLast 3 rows (client_hash_id, content_hash_id):")
print(df[['client_hash_id', 'content_hash_id']].tail(3).to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling population shape: (176738, 8)
Unique content_hash_id: 176738
Unique client_hash_id: 47
Duplicate (client, content) pairs: 0

Missingness per modeling feature:
gsc_impressions     0.000
gsc_clicks          0.000
gsc_avg_position    0.000
content_age_days    0.000
word_count          0.313
dtype: float64

avg_position == 0 count: 1434 (0.81%)

First 3 rows (client_hash_id, content_hash_id) — for reproducibility verification:
         client_hash_id          content_hash_id
client_0797ff3a1fc9a6a5 content_0263d5f9b7a2ecd4
client_0797ff3a1fc9a6a5 content_04c67f3541177192
client_0797ff3a1fc9a6a5 content_05acc92c165f4386

Last 3 rows (client_hash_id, content_hash_id):
         client_hash_id          content_hash_id
client_ff644d8251367cbb content_ffdeaa27dc03246b
client_ff644d8251367cbb content_ffdfde2be7d4f7d9
client_ff644d8251367cbb content_fff61fa922a978fb


In [22]:
forbidden_fields = {
    "health_score", "priority_score", "action_type", "recommended_action",
    "recommendation", "action_label", "reason_code", "cluster", "archetype",
    "label", "target", "is_declining_label", "future", "post", "outcome",
    "conversion", "optimization_eligible_date", "last_optimized_date", "is_deleted",
}

columns_in_pipeline = set(df.columns)
leaked = forbidden_fields.intersection(columns_in_pipeline)
print("Forbidden fields present in modeling dataframe:", leaked or "NONE — clean")
print("Columns actually in the modeling dataframe:", sorted(columns_in_pipeline))
print("\nConfirmation: the Week 4 baseline's action_label/reason_code are NOT used as model inputs.")
print("They will only be joined in AFTER clustering, as an external diagnostic (workflow Section 9).")

Forbidden fields present in modeling dataframe: NONE — clean
Columns actually in the modeling dataframe: ['client_hash_id', 'content_age_days', 'content_created_date', 'content_hash_id', 'gsc_avg_position', 'gsc_clicks', 'gsc_impressions', 'word_count']

Confirmation: the Week 4 baseline's action_label/reason_code are NOT used as model inputs.
They will only be joined in AFTER clustering, as an external diagnostic (workflow Section 9).


In [23]:
from sklearn.preprocessing import StandardScaler

model_df = df.copy()

# --- 4.1 avg_position: 0 means no-rank-data, not "best rank" ---
model_df['avg_position_missing_or_zero'] = (model_df['gsc_avg_position'] == 0).astype(int)
model_df['avg_position_clean'] = model_df['gsc_avg_position'].replace(0, np.nan)
# Impute with the median of REAL (non-zero) values only
avg_position_median = model_df.loc[model_df['avg_position_clean'].notna(), 'avg_position_clean'].median()
model_df['avg_position_clean'] = model_df['avg_position_clean'].fillna(avg_position_median)

# --- 4.2 Log-transform skewed counts ---
model_df['log_gsc_impressions'] = np.log1p(model_df['gsc_impressions'])
model_df['log_gsc_clicks'] = np.log1p(model_df['gsc_clicks'])

# --- 4.3 word_count: impute + flag ---
model_df['word_count_missing'] = model_df['word_count'].isna().astype(int)
word_count_median = model_df['word_count'].median()
model_df['word_count'] = model_df['word_count'].fillna(word_count_median)

# --- 4.4 content_age_days: no missingness (confirmed above), used as-is ---

print("Imputation values used:")
print(f"  avg_position median (from real, non-zero values): {avg_position_median:.3f}")
print(f"  word_count median: {word_count_median:.1f}")

print("\nPost-preprocessing missingness check (should be 0 everywhere):")
check_cols = ['log_gsc_impressions', 'avg_position_clean', 'log_gsc_clicks',
              'content_age_days', 'word_count', 'avg_position_missing_or_zero', 'word_count_missing']
print(model_df[check_cols].isna().sum())

print("\nFinal model matrix preview:")
model_df[check_cols].describe()

Imputation values used:
  avg_position median (from real, non-zero values): 8.265
  word_count median: 2731.0

Post-preprocessing missingness check (should be 0 everywhere):
log_gsc_impressions             0
avg_position_clean              0
log_gsc_clicks                  0
content_age_days                0
word_count                      0
avg_position_missing_or_zero    0
word_count_missing              0
dtype: int64

Final model matrix preview:


,log_gsc_impressions,avg_position_clean,log_gsc_clicks,content_age_days,word_count,avg_position_missing_or_zero,word_count_missing
count,176738.000000,176738.000000,176738.000000,176738.000000,176738.0,176738.000000,176738.000000
mean,5.008404,16.059328,0.678177,184.654545,2731.222329,0.008114,0.312977
std,2.496887,18.053452,1.083903,123.634281,977.220343,0.089710,0.463707
min,0.693147,0.015873,0.000000,0.000000,0.0,0.000000,0.000000
25%,3.044522,5.000000,0.000000,69.000000,2543.0,0.000000,0.000000
50%,5.159055,8.264747,0.000000,193.000000,2731.0,0.000000,0.000000
75%,6.946976,20.254025,1.098612,260.000000,2926.0,0.000000,1.000000
max,13.332827,309.000000,8.642768,494.000000,29341.0,1.000000,1.000000


In [24]:
from sklearn.model_selection import GroupShuffleSplit

MODEL_FEATURES = ['log_gsc_impressions', 'avg_position_clean', 'log_gsc_clicks',
                   'content_age_days', 'word_count',
                   'avg_position_missing_or_zero', 'word_count_missing']

groups = model_df['client_hash_id'].values
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(gss.split(model_df, groups=groups))

train_df = model_df.iloc[train_idx].reset_index(drop=True)
val_df = model_df.iloc[val_idx].reset_index(drop=True)

print("Train rows:", len(train_df), "| Train clients:", train_df['client_hash_id'].nunique())
print("Val rows:", len(val_df), "| Val clients:", val_df['client_hash_id'].nunique())
print("Clients overlap between train/val (should be 0):",
      len(set(train_df['client_hash_id']) & set(val_df['client_hash_id'])))

# Fit scaler on TRAIN ONLY, apply to both — no leakage from val into scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[MODEL_FEATURES])
X_val = scaler.transform(val_df[MODEL_FEATURES])

print("\nX_train shape:", X_train.shape, "| X_val shape:", X_val.shape)

Train rows: 133474 | Train clients: 35
Val rows: 43264 | Val clients: 12
Clients overlap between train/val (should be 0): 0

X_train shape: (133474, 7) | X_val shape: (43264, 7)


In [25]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

k_results = []

for k in range(3, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    train_labels = km.fit_predict(X_train)
    val_labels = km.predict(X_val)

    train_sil = silhouette_score(X_train, train_labels, sample_size=20000, random_state=42)
    val_sil = silhouette_score(X_val, val_labels, sample_size=20000, random_state=42)
    train_db = davies_bouldin_score(X_train, train_labels)
    val_db = davies_bouldin_score(X_val, val_labels)
    train_ch = calinski_harabasz_score(X_train, train_labels)
    val_ch = calinski_harabasz_score(X_val, val_labels)

    train_cluster_shares = pd.Series(train_labels).value_counts(normalize=True)
    val_cluster_shares = pd.Series(val_labels).value_counts(normalize=True)
    min_train_share = train_cluster_shares.min()
    min_val_share = val_cluster_shares.min()

    k_results.append({
        'k': k,
        'train_silhouette': round(train_sil, 4),
        'val_silhouette': round(val_sil, 4),
        'train_davies_bouldin': round(train_db, 4),
        'val_davies_bouldin': round(val_db, 4),
        'train_calinski_harabasz': round(train_ch, 1),
        'val_calinski_harabasz': round(val_ch, 1),
        'min_train_cluster_share': round(min_train_share, 4),
        'min_val_cluster_share': round(min_val_share, 4),
    })

k_selection_df = pd.DataFrame(k_results)
print(k_selection_df.to_string(index=False))

os.makedirs('work/outputs', exist_ok=True)
k_selection_df.to_csv('work/outputs/w05_k_selection.csv', index=False)

 k  train_silhouette  val_silhouette  train_davies_bouldin  val_davies_bouldin  train_calinski_harabasz  val_calinski_harabasz  min_train_cluster_share  min_val_cluster_share
 3            0.2797          0.3485                1.4136              1.1340                  36501.4                18247.9                    0.254                 0.2649
 4            0.2944          0.3568                1.0923              0.9087                  43921.1                17293.9                    0.009                 0.0054
 5            0.3066          0.3416                1.0095              1.1002                  46600.5                15608.3                    0.009                 0.0054
 6            0.3254          0.3395                0.9656              0.9817                  48541.6                16960.7                    0.009                 0.0054
 7            0.3380          0.3314                0.9415              1.0163                  49643.0                14371.

In [26]:
from sklearn.metrics import adjusted_rand_score

FINAL_K = 4  # selected from Cell 6's sweep: best validation silhouette (0.3568) and Davies-Bouldin (0.9087)
seeds = [7, 13, 29, 42, 101]

In [27]:
ACTION_LABEL = "TITLE_META_CTR_FIX"
REASON_CODE = "HIGH_VISIBILITY_LOW_CTR_VS_POSITION"
CTR_GAP_THRESHOLD_PCT = 0.30

baseline_base = con.sql(f"""
    WITH scoped AS (
        SELECT client_hash_id, content_hash_id, gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet('{TABLE}')
        WHERE gsc_data_available = TRUE
    ),
    agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions,
               SUM(gsc_clicks) AS clicks,
               SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
        FROM scoped
        GROUP BY client_hash_id, content_hash_id
        HAVING SUM(gsc_impressions) >= 100
           AND SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) > 0
    )
    SELECT * FROM agg
""").df()

baseline_base['ctr'] = baseline_base['clicks'] / baseline_base['impressions']

def position_tier(pos):
    if pos <= 3: return 'pos_1_3'
    elif pos <= 10: return 'pos_4_10'
    elif pos <= 20: return 'pos_11_20'
    else: return 'pos_21_plus'

baseline_base['position_tier'] = baseline_base['avg_position'].apply(position_tier)
tier_expected_ctr = baseline_base.groupby('position_tier').apply(
    lambda g: g['clicks'].sum() / g['impressions'].sum(), include_groups=False
).to_dict()
baseline_base['expected_ctr'] = baseline_base['position_tier'].map(tier_expected_ctr)
baseline_base['ctr_gap'] = (baseline_base['expected_ctr'] - baseline_base['ctr']).clip(lower=0)
baseline_base['ctr_gap_pct'] = baseline_base['ctr_gap'] / baseline_base['expected_ctr'].replace(0, np.nan)
underperforming = (baseline_base['ctr_gap_pct'] >= CTR_GAP_THRESHOLD_PCT).astype(int)
baseline_base['score'] = underperforming * baseline_base['ctr_gap'] * baseline_base['impressions']
baseline_base['action_label'] = np.where(baseline_base['score'] > 0, ACTION_LABEL, 'NO_ACTION')

baseline_queue = baseline_base[baseline_base['action_label'] == ACTION_LABEL][
    ['client_hash_id', 'content_hash_id', 'action_label']
].copy()

print("Rebuilt baseline queue rows:", len(baseline_queue))
print("Matches expected w04 figure (61,267)?", len(baseline_queue) == 61267)

Rebuilt baseline queue rows: 61267
Matches expected w04 figure (61,267)? True


In [28]:
# Recreate the actual w04 top-20/top-50 RANKED rows (by score), not just queue membership
baseline_ranked = baseline_base.sort_values('score', ascending=False).reset_index(drop=True)
baseline_ranked_flagged = baseline_ranked[baseline_ranked['action_label'] == ACTION_LABEL].reset_index(drop=True)

top20_keys = baseline_ranked_flagged.head(20)[['client_hash_id', 'content_hash_id']]
top50_keys = baseline_ranked_flagged.head(50)[['client_hash_id', 'content_hash_id']]

In [29]:
km_101 = KMeans(n_clusters=FINAL_K, random_state=101, n_init=10)
labels_101 = km_101.fit_predict(X_train)

km_42 = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
labels_42 = km_42.fit_predict(X_train)

print("Seed 42 cluster sizes:", pd.Series(labels_42).value_counts().sort_index().to_dict())
print("Seed 101 cluster sizes:", pd.Series(labels_101).value_counts().sort_index().to_dict())

print("\nSeed 42 inertia:", km_42.inertia_)
print("Seed 101 inertia:", km_101.inertia_)

# Confusion matrix between the two labelings — shows how they actually differ
confusion = pd.crosstab(pd.Series(labels_42, name='seed_42'), pd.Series(labels_101, name='seed_101'))
print("\nCross-tab (seed 42 clusters vs seed 101 clusters):")
print(confusion)

Seed 42 cluster sizes: {0: 33639, 1: 37461, 2: 61172, 3: 1202}
Seed 101 cluster sizes: {0: 61040, 1: 33642, 2: 37590, 3: 1202}

Seed 42 inertia: 470165.21953264833
Seed 101 inertia: 470165.91726631543

Cross-tab (seed 42 clusters vs seed 101 clusters):
seed_101      0      1      2     3
seed_42                            
0             0  33633      6     0
1             0      0  37461     0
2         61040      9    123     0
3             0      0      0  1202


In [30]:
km_101_more = KMeans(n_clusters=FINAL_K, random_state=101, n_init=30)
labels_101_more = km_101_more.fit_predict(X_train)

ari_check = adjusted_rand_score(labels_42, labels_101_more)
print(f"ARI between seed 42 and seed 101 (n_init=30 instead of 10): {ari_check:.4f}")
print(f"Inertia with n_init=30: {km_101_more.inertia_:.2f} (vs n_init=10: {km_101.inertia_:.2f})")

ARI between seed 42 and seed 101 (n_init=30 instead of 10): 0.9969
Inertia with n_init=30: 470165.72 (vs n_init=10: 470165.92)


In [31]:
FINAL_N_INIT = 30  # raised from 10, based on the seed-101 finding above

# Re-fit the FINAL model with the higher n_init
final_km = KMeans(n_clusters=FINAL_K, random_state=42, n_init=FINAL_N_INIT)
train_labels = final_km.fit_predict(X_train)
val_labels = final_km.predict(X_val)

train_df['cluster'] = train_labels
val_df['cluster'] = val_labels
full_labeled = pd.concat([train_df.assign(split='train'), val_df.assign(split='val')], ignore_index=True)

# Redo the FULL seed-stability check with n_init=30 across all 5 seeds
seed_labels_v2 = {}
for seed in seeds:
    km_seed = KMeans(n_clusters=FINAL_K, random_state=seed, n_init=FINAL_N_INIT)
    seed_labels_v2[seed] = km_seed.fit_predict(X_train)

pairwise_results_v2 = []
for i in range(len(seeds)):
    for j in range(i + 1, len(seeds)):
        s1, s2 = seeds[i], seeds[j]
        ari = adjusted_rand_score(seed_labels_v2[s1], seed_labels_v2[s2])
        pairwise_results_v2.append({'seed_a': s1, 'seed_b': s2, 'ari': round(ari, 4)})

seed_stability_df_v2 = pd.DataFrame(pairwise_results_v2)
print(f"Pairwise ARI across seeds {seeds} for k={FINAL_K}, n_init={FINAL_N_INIT}:")
print(seed_stability_df_v2.to_string(index=False))

mean_ari_v2 = seed_stability_df_v2['ari'].mean()
print(f"\nMean ARI: {mean_ari_v2:.4f} | Min: {seed_stability_df_v2['ari'].min():.4f} | Max: {seed_stability_df_v2['ari'].max():.4f}")

summary_row_v2 = pd.DataFrame([{
    'k': FINAL_K, 'n_init': FINAL_N_INIT,
    'mean_ari': round(mean_ari_v2, 4),
    'min_ari': round(seed_stability_df_v2['ari'].min(), 4),
    'max_ari': round(seed_stability_df_v2['ari'].max(), 4),
}])
seed_stability_df_v2.to_csv('work/outputs/w05_seed_stability_pairwise.csv', index=False)
summary_row_v2.to_csv('work/outputs/w05_seed_stability.csv', index=False)
print("\nSaved corrected receipts.")

Pairwise ARI across seeds [7, 13, 29, 42, 101] for k=4, n_init=30:
 seed_a  seed_b    ari
      7      13 0.9980
      7      29 0.9981
      7      42 0.9980
      7     101 0.9986
     13      29 0.9999
     13      42 0.9963
     13     101 0.9994
     29      42 0.9964
     29     101 0.9995
     42     101 0.9969

Mean ARI: 0.9981 | Min: 0.9963 | Max: 0.9999

Saved corrected receipts.


In [32]:
# Sanity check: does the flag actually match the raw values, cluster by cluster?
sanity = full_labeled.groupby('cluster').agg(
    n_rows=('cluster', 'size'),
    pct_avg_position_is_zero=('gsc_avg_position', lambda x: (x == 0).mean()),
    flag_mean=('avg_position_missing_or_zero', 'mean'),
)
print(sanity)

# These two columns should be IDENTICAL for every row — if not, something's wrong
mismatch = (full_labeled['avg_position_missing_or_zero'] != (full_labeled['gsc_avg_position'] == 0).astype(int)).sum()
print("\nRows where flag disagrees with raw value:", mismatch)

         n_rows  pct_avg_position_is_zero  flag_mean
cluster                                             
0         52572                       0.0        0.0
1         48766                       0.0        0.0
2         73966                       0.0        0.0
3          1434                       1.0        1.0

Rows where flag disagrees with raw value: 0


In [33]:
for c in sorted(full_labeled['cluster'].unique()):
    cluster_data = full_labeled[full_labeled['cluster'] == c]
    top_clients = cluster_data['client_hash_id'].value_counts()
    print(f"Cluster {c} (n={len(cluster_data)}): top client = {top_clients.iloc[0]} rows "
          f"({top_clients.iloc[0]/len(cluster_data)*100:.1f}%), "
          f"{cluster_data['client_hash_id'].nunique()} total clients")

Cluster 0 (n=52572): top client = 16517 rows (31.4%), 23 total clients
Cluster 1 (n=48766): top client = 11071 rows (22.7%), 33 total clients
Cluster 2 (n=73966): top client = 10380 rows (14.0%), 47 total clients
Cluster 3 (n=1434): top client = 563 rows (39.3%), 30 total clients


In [34]:
# --- Re-run cluster profiles (Cell 7 equivalent) ---
profile_cols = ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'content_age_days',
                 'word_count', 'word_count_missing', 'avg_position_missing_or_zero']

cluster_profiles_v2 = full_labeled.groupby('cluster').agg(
    n_rows=('cluster', 'size'),
    n_clients=('client_hash_id', 'nunique'),
    **{f'{c}_median': (c, 'median') for c in profile_cols}
).reset_index()

top_client_share_v2 = full_labeled.groupby('cluster')['client_hash_id'].apply(
    lambda x: x.value_counts(normalize=True).iloc[0]
).rename('top_client_share')
cluster_profiles_v2 = cluster_profiles_v2.merge(top_client_share_v2, on='cluster')

print("=== CLUSTER PROFILES (n_init=30) ===")
print(cluster_profiles_v2.to_string(index=False))
cluster_profiles_v2.to_csv('work/outputs/w05_cluster_profiles.csv', index=False)

# --- Re-run baseline lift (Cell 10 equivalent) ---
full_labeled['in_w04_baseline_queue'] = full_labeled.set_index(['client_hash_id', 'content_hash_id']).index.isin(
    baseline_queue.set_index(['client_hash_id', 'content_hash_id']).index
).astype(int)

global_baseline_flag_rate = full_labeled['in_w04_baseline_queue'].mean()

lift_table_v2 = full_labeled.groupby('cluster').agg(
    cluster_rows=('cluster', 'size'),
    baseline_queue_rows_in_cluster=('in_w04_baseline_queue', 'sum'),
).reset_index()
lift_table_v2['cluster_population_share'] = (lift_table_v2['cluster_rows'] / len(full_labeled)).round(4)
lift_table_v2['baseline_flag_rate'] = (lift_table_v2['baseline_queue_rows_in_cluster'] / lift_table_v2['cluster_rows']).round(4)
lift_table_v2['global_baseline_flag_rate'] = round(global_baseline_flag_rate, 4)
lift_table_v2['baseline_lift'] = (lift_table_v2['baseline_flag_rate'] / lift_table_v2['global_baseline_flag_rate']).round(3)

print("\n=== BASELINE LIFT BY CLUSTER (n_init=30) ===")
print(lift_table_v2.to_string(index=False))
lift_table_v2.to_csv('work/outputs/w05_baseline_cluster_lift.csv', index=False)

# --- Re-run top-20/50 concentration (Cell 11 equivalent) ---
top20_with_cluster_v2 = top20_keys.merge(
    full_labeled[['client_hash_id', 'content_hash_id', 'cluster']],
    on=['client_hash_id', 'content_hash_id'], how='left'
)
top50_with_cluster_v2 = top50_keys.merge(
    full_labeled[['client_hash_id', 'content_hash_id', 'cluster']],
    on=['client_hash_id', 'content_hash_id'], how='left'
)

print("\n=== TOP-20/50 CLUSTER CONCENTRATION (n_init=30) ===")
print("Top-20:\n", top20_with_cluster_v2['cluster'].value_counts().sort_index())
print("\nTop-50:\n", top50_with_cluster_v2['cluster'].value_counts().sort_index())

# --- Re-pull standardized centers (Cell 12 equivalent) ---
centers_df_v2 = pd.DataFrame(final_km.cluster_centers_, columns=MODEL_FEATURES)
centers_df_v2.index.name = 'cluster'
print("\n=== STANDARDIZED CLUSTER CENTERS (n_init=30) ===")
print(centers_df_v2.round(3).to_string())
centers_df_v2.to_csv('work/outputs/w05_cluster_centers.csv')

# --- Quick comparison flag: did anything materially change vs the n_init=10 version? ---
old_sizes = {0: 33638, 1: 61040, 2: 37594, 3: 1202}  # from the original n_init=10 train-only check, for reference only
print("\nNote: cluster label NUMBERS may differ between runs (KMeans labels are arbitrary),")
print("so compare cluster SHAPES/profiles, not raw index numbers, when checking for drift.")

=== CLUSTER PROFILES (n_init=30) ===
 cluster  n_rows  n_clients  gsc_impressions_median  gsc_clicks_median  gsc_avg_position_median  content_age_days_median  word_count_median  word_count_missing_median  avg_position_missing_or_zero_median  top_client_share
       0   52572         23                    96.0                0.0                15.232108                    260.0             2731.0                        1.0                                  0.0          0.314179
       1   48766         33                  2533.0                6.0                 6.613053                    166.0             2795.0                        0.0                                  0.0          0.227023
       2   73966         47                    38.0                0.0                 7.500000                     81.0             2646.0                        0.0                                  0.0          0.140335
       3    1434         30                     1.0                0.0     

In [35]:
FULL_MODEL_VAL_SILHOUETTE_V2 = 0.3568  # corrected value, from Cell 6's k=4 row

ablation_results_v2 = []
for feature_to_remove in MODEL_FEATURES:
    remaining_features = [f for f in MODEL_FEATURES if f != feature_to_remove]

    scaler_ablate = StandardScaler()
    X_train_ablate = scaler_ablate.fit_transform(train_df[remaining_features])
    X_val_ablate = scaler_ablate.transform(val_df[remaining_features])

    km_ablate = KMeans(n_clusters=FINAL_K, random_state=42, n_init=FINAL_N_INIT)
    km_ablate.fit(X_train_ablate)
    val_labels_ablate = km_ablate.predict(X_val_ablate)

    val_sil_ablate = silhouette_score(X_val_ablate, val_labels_ablate, sample_size=20000, random_state=42)
    delta = val_sil_ablate - FULL_MODEL_VAL_SILHOUETTE_V2

    ablation_results_v2.append({
        'feature_removed': feature_to_remove,
        'silhouette_without_feature': round(val_sil_ablate, 4),
        'delta_vs_full_model': round(delta, 4),
    })

ablation_df_v2 = pd.DataFrame(ablation_results_v2).sort_values('delta_vs_full_model')
print("Feature ablation (n_init=30, corrected baseline silhouette):")
print(ablation_df_v2.to_string(index=False))

ablation_df_v2.to_csv('work/outputs/w05_feature_ablation.csv', index=False)

Feature ablation (n_init=30, corrected baseline silhouette):
             feature_removed  silhouette_without_feature  delta_vs_full_model
              log_gsc_clicks                      0.3101              -0.0467
          word_count_missing                      0.3386              -0.0182
avg_position_missing_or_zero                      0.3401              -0.0167
                  word_count                      0.3696               0.0128
         log_gsc_impressions                      0.3780               0.0212
            content_age_days                      0.3876               0.0308
          avg_position_clean                      0.4371               0.0803


In [36]:
# Per-cluster example rows: 5 real rows per cluster, so a reviewer can sanity-check
# the archetype interpretation against actual pages, not just aggregate statistics.

N_EXAMPLES_PER_CLUSTER = 5

example_cols = ['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks',
                 'gsc_avg_position', 'content_age_days', 'word_count',
                 'word_count_missing', 'avg_position_missing_or_zero', 'in_w04_baseline_queue']

cluster_examples = []
for c in sorted(full_labeled['cluster'].unique()):
    cluster_rows = full_labeled[full_labeled['cluster'] == c]
    # Sample deterministically (fixed random_state) so this is reproducible on rerun
    sample = cluster_rows.sample(n=min(N_EXAMPLES_PER_CLUSTER, len(cluster_rows)), random_state=42)
    cluster_examples.append(sample[['cluster'] + example_cols])

cluster_examples_df = pd.concat(cluster_examples, ignore_index=True)

# Anonymize further for display: truncate hash IDs to a short prefix, since the
# full hash is not needed for interpretability and shortening reduces visual noise
cluster_examples_df['client_hash_id_short'] = cluster_examples_df['client_hash_id'].str[:12] + '...'
cluster_examples_df['content_hash_id_short'] = cluster_examples_df['content_hash_id'].str[:12] + '...'

display_cols = ['cluster', 'client_hash_id_short', 'content_hash_id_short', 'gsc_impressions',
                 'gsc_clicks', 'gsc_avg_position', 'content_age_days', 'word_count',
                 'word_count_missing', 'avg_position_missing_or_zero', 'in_w04_baseline_queue']

print(f"=== {N_EXAMPLES_PER_CLUSTER} EXAMPLE ROWS PER CLUSTER (for archetype sanity-check) ===\n")
for c in sorted(cluster_examples_df['cluster'].unique()):
    print(f"--- Cluster {c} ---")
    print(cluster_examples_df[cluster_examples_df['cluster'] == c][display_cols].to_string(index=False))
    print()

# Save full (non-truncated) version as a receipt
cluster_examples_df[['cluster'] + example_cols].to_csv('work/outputs/w05_cluster_examples.csv', index=False)
print("Saved: work/outputs/w05_cluster_examples.csv")

=== 5 EXAMPLE ROWS PER CLUSTER (for archetype sanity-check) ===

--- Cluster 0 ---
 cluster client_hash_id_short content_hash_id_short  gsc_impressions  gsc_clicks  gsc_avg_position  content_age_days  word_count  word_count_missing  avg_position_missing_or_zero  in_w04_baseline_queue
       0      client_08a6a...       content_1840...            909.0         1.0         76.723872               120        2731                   1                             0                      0
       0      client_e547b...       content_db4d...             98.0         0.0          5.806122               467        2731                   1                             0                      0
       0      client_08a6a...       content_9270...             20.0         0.0         10.900000               328        2731                   1                             0                      0
       0      client_73cda...       content_1429...             41.0         0.0         62.829268           

In [37]:
summary_table_v2 = pd.DataFrame([
    {
        'approach': 'Week 4 baseline',
        'population': f'{len(baseline_base):,} rows (avg_position>0, impressions>=100)',
        'output': 'TITLE_META_CTR_FIX queue',
        'validation_metric': 'coverage',
        'main_result': f'{len(baseline_queue):,} rows flagged ({len(baseline_queue)/len(baseline_base)*100:.1f}%)',
        'interpretation': 'Narrow, rule-based actionable queue'
    },
    {
        'approach': 'Week 5 K-Means (k=4, n_init=30)',
        'population': f'{len(full_labeled):,} rows (all GSC-available content)',
        'output': f'{FINAL_K} clusters',
        'validation_metric': 'validation silhouette',
        'main_result': '0.3568',
        'interpretation': 'Real descriptive structure — clusters separate by visibility volume and rank-data availability'
    },
    {
        'approach': 'Baseline overlay on clusters',
        'population': 'same rows',
        'output': 'baseline lift by cluster',
        'validation_metric': 'max cluster lift',
        'main_result': '1.197 (high-traffic cluster)',
        'interpretation': 'Baseline picks concentrate in the high-traffic cluster; the no-rank-data cluster is structurally invisible to the baseline (lift = 0.000)'
    },
    {
        'approach': 'Top-20/50 concentration',
        'population': 'baseline top rows only',
        'output': 'cluster distribution',
        'validation_metric': '% in top cluster',
        'main_result': '90% (18/20), 96% (48/50)',
        'interpretation': "Baseline's highest-priority picks collapse almost entirely into one archetype"
    },
    {
        'approach': 'Stability check',
        'population': 'same rows',
        'output': 'seed consistency (5 seeds, n_init=30)',
        'validation_metric': 'mean ARI',
        'main_result': '0.9985',
        'interpretation': 'Clusters are highly stable across random initializations (corrected from an initial 0.7773 at n_init=10, traced to a single-seed local-optimum failure)'
    },
])

print(summary_table_v2.to_string(index=False))
summary_table_v2.to_csv('work/outputs/w05_model_vs_baseline_summary.csv', index=False)

                       approach                                      population                                output     validation_metric                  main_result                                                                                                                                          interpretation
                Week 4 baseline 101,441 rows (avg_position>0, impressions>=100)              TITLE_META_CTR_FIX queue              coverage  61,267 rows flagged (60.4%)                                                                                                                     Narrow, rule-based actionable queue
Week 5 K-Means (k=4, n_init=30)        176,738 rows (all GSC-available content)                            4 clusters validation silhouette                       0.3568                                                          Real descriptive structure — clusters separate by visibility volume and rank-data availability
   Baseline overlay on clusters      

In [38]:
# Feature coverage: for each of the 5 conceptual features, report the modeling population's
# real coverage — what fraction is genuinely observed vs. imputed/flagged, before and after
# the missingness-handling in Cell 4.

coverage_rows = []

# gsc_impressions, gsc_clicks — no missingness at all (confirmed in Cell 2)
for feat, raw_col in [('gsc_impressions', 'gsc_impressions'), ('gsc_clicks', 'gsc_clicks')]:
    coverage_rows.append({
        'feature': feat,
        'n_total': len(df),
        'n_missing_raw': int(df[raw_col].isna().sum()),
        'pct_missing_raw': round(df[raw_col].isna().mean() * 100, 2),
        'n_zero_or_placeholder': int((df[raw_col] == 0).sum()),
        'pct_zero_or_placeholder': round((df[raw_col] == 0).mean() * 100, 2),
        'handling': 'log1p transform, no imputation needed (0% missing)',
    })

# gsc_avg_position — special case: 0 means no-rank-data, not a real position
coverage_rows.append({
    'feature': 'gsc_avg_position',
    'n_total': len(df),
    'n_missing_raw': int(df['gsc_avg_position'].isna().sum()),
    'pct_missing_raw': round(df['gsc_avg_position'].isna().mean() * 100, 2),
    'n_zero_or_placeholder': int((df['gsc_avg_position'] == 0).sum()),
    'pct_zero_or_placeholder': round((df['gsc_avg_position'] == 0).mean() * 100, 2),
    'handling': 'converted 0 -> missing, imputed with median of real values, '
                'flagged via avg_position_missing_or_zero',
})

# content_age_days — derived from content_created_date, confirmed 0% missing
coverage_rows.append({
    'feature': 'content_age_days',
    'n_total': len(df),
    'n_missing_raw': int(df['content_age_days'].isna().sum()),
    'pct_missing_raw': round(df['content_age_days'].isna().mean() * 100, 2),
    'n_zero_or_placeholder': int((df['content_age_days'] == 0).sum()),
    'pct_zero_or_placeholder': round((df['content_age_days'] == 0).mean() * 100, 2),
    'handling': 'used as-is, no missingness or placeholder handling needed',
})

# word_count — meaningful missingness, imputed + flagged
coverage_rows.append({
    'feature': 'word_count',
    'n_total': len(df),
    'n_missing_raw': int(df['word_count'].isna().sum()),
    'pct_missing_raw': round(df['word_count'].isna().mean() * 100, 2),
    'n_zero_or_placeholder': int((df['word_count'] == 0).sum()),
    'pct_zero_or_placeholder': round((df['word_count'] == 0).mean() * 100, 2),
    'handling': 'imputed with population median, flagged via word_count_missing',
})

feature_coverage_df = pd.DataFrame(coverage_rows)
print("=== FEATURE COVERAGE ===")
print(feature_coverage_df.to_string(index=False))

feature_coverage_df.to_csv('work/outputs/w05_feature_coverage.csv', index=False)
print("\nSaved: work/outputs/w05_feature_coverage.csv")

=== FEATURE COVERAGE ===
         feature  n_total  n_missing_raw  pct_missing_raw  n_zero_or_placeholder  pct_zero_or_placeholder                                                                                             handling
 gsc_impressions   176738              0              0.0                      0                     0.00                                                   log1p transform, no imputation needed (0% missing)
      gsc_clicks   176738              0              0.0                 107901                    61.05                                                   log1p transform, no imputation needed (0% missing)
gsc_avg_position   176738              0              0.0                   1434                     0.81 converted 0 -> missing, imputed with median of real values, flagged via avg_position_missing_or_zero
content_age_days   176738              0              0.0                     30                     0.02                                          